In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

# Huggingface Hub Integration

In [2]:
from langchain_huggingface import HuggingFaceEndpoint

/home/james/development/gen-ai-course/venv/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [3]:
repo_id = "mistralai/Mistral-7B-Instruct-v0.3"

# Try using HuggingFaceEndpoint with streaming disabled
llm = HuggingFaceEndpoint(
    repo_id=repo_id,
    temperature=0.7,
    max_new_tokens=150,  # Changed from max_length to max_new_tokens
    huggingfacehub_api_token=os.getenv("HF_TOKEN"),
    streaming=False  # Disable streaming to avoid the post method issue
)

/home/james/development/gen-ai-course/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
llm.invoke("What is machine learning?")


ValueError: Model mistralai/Mistral-7B-Instruct-v0.3 is not supported for task text-generation and provider novita. Supported task: conversational.

In [5]:
# Alternative approach: Use HuggingFace InferenceClient directly
from huggingface_hub import InferenceClient
from langchain.llms.base import LLM
from typing import Optional, List, Any
from pydantic import Field

class MistralLLM(LLM):
    client: InferenceClient = Field(default=None)
    model_name: str = "mistralai/Mistral-7B-Instruct-v0.3"
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.client = InferenceClient(
            model=self.model_name,
            token=os.getenv("HF_TOKEN")
        )
    
    @property
    def _llm_type(self) -> str:
        return "mistral"
    
    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Optional[Any] = None,
        **kwargs
    ) -> str:
        # Use conversational API for Mistral
        messages = [{"role": "user", "content": prompt}]
        response = self.client.chat_completion(
            messages=messages,
            max_tokens=150,
            temperature=0.7
        )
        return response.choices[0].message.content

# Create the custom LLM
mistral_llm = MistralLLM()

In [6]:
# Test the custom Mistral LLM
response = mistral_llm.invoke("What is machine learning?")
print(response)

Machine learning is a subset of artificial intelligence that enables a computer or machine to learn from data, without being explicitly programmed. It involves the development of algorithms that allow the machine to identify patterns in the data and make decisions or predictions based on those patterns.



In [7]:
from langchain import PromptTemplate, LLMChain

template = """
Question: {question}
Answer: Let's think step by step.
"""

prompt = PromptTemplate(
    input_variables=["question"],
    template=template
)
print(prompt)

input_variables=['question'] input_types={} partial_variables={} template="\nQuestion: {question}\nAnswer: Let's think step by step.\n"


In [8]:
llm_chain = LLMChain(
    llm=mistral_llm,
    prompt=prompt
)
llm_chain.invoke({"question": "What is machine learning?"})

/tmp/ipykernel_594871/3779631490.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(


{'question': 'What is machine learning?',
 'text': 'Machine learning is a subset of artificial intelligence (AI) that allows computers to learn from data, without being explicitly programmed. It involves training algorithms on large datasets, which enables them to make predictions or decisions based on new, unseen data.\n'}

In [13]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import torch

model_name = "BAAI/bge-small-en-v1.5"
model_kwargs = {"device": "cuda" if torch.cuda.is_available() else "cpu"}
encode_kwargs = {"normalize_embeddings": True}
hf = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

In [14]:
embedding = hf.embed_query("Hi this is Harrison")

In [15]:
embedding

[-0.029063696041703224,
 -0.008915022015571594,
 0.03558390960097313,
 -0.06222883611917496,
 0.0024855437222868204,
 -0.023602532222867012,
 0.07593873143196106,
 0.008911673910915852,
 0.018002599477767944,
 0.0021945894695818424,
 0.010421132668852806,
 -0.07986357063055038,
 0.007148312404751778,
 0.02014848403632641,
 0.01686391606926918,
 -0.06320749968290329,
 0.06079493835568428,
 -0.02142365649342537,
 -0.01935598812997341,
 -0.02760784886777401,
 0.03864438831806183,
 0.03962988033890724,
 -0.052514463663101196,
 -0.05342833697795868,
 0.04681415855884552,
 -0.005146898329257965,
 0.00995529256761074,
 -0.013670136220753193,
 -0.03345843404531479,
 -0.08511195331811905,
 -0.023732401430606842,
 0.010551846586167812,
 0.028330475091934204,
 0.02791464328765869,
 0.00775937782600522,
 0.0010389420203864574,
 -0.0027399230748414993,
 0.03693924471735954,
 -0.026221195235848427,
 0.012939524836838245,
 -0.022442294284701347,
 -0.05161753669381142,
 0.01095594558864832,
 0.0398739